In [ ]:
train_sets = [

    '../data/raw/sample_23.csv',

    '../data/processed/sample_23_znorm.parquet'

]

test_set = '../data/raw/sample_24.csv'



# Set to None to use full dataset

SAMPLE_SIZE = None

CLUSTERS_COUNT = 7

RANDOM_STATE = 42



# Runtime controls

PLOT_INDIVIDUAL_SERIES = False

MAX_SERIES_PER_CLUSTER_PLOT = 300

KMEDOID_MAX_ITER = 60

CONSENSUS_RUNS = 8

CONSENSUS_SAMPLE_RATIO = 0.85



#%pip install matplotlib

#%pip install pandas

#%pip install scikit-learn

In [ ]:
import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.cluster import AgglomerativeClustering

In [ ]:
_DATA_CACHE = {}



def load_values(filepath):

    """Load ids and numeric matrix (rows x timesteps) from csv/parquet."""

    if filepath in _DATA_CACHE:

        return _DATA_CACHE[filepath]



    path = Path(filepath)

    if path.suffix.lower() == '.parquet':

        df = pd.read_parquet(path)

    else:

        df = pd.read_csv(path)



    ids = df.iloc[:, 0].to_numpy()

    headers = df.columns[1:].tolist()

    values = df.iloc[:, 1:].to_numpy(dtype=np.float32, copy=False)



    _DATA_CACHE[filepath] = (ids, headers, values)

    return ids, headers, values

In [ ]:
def calculate_copy_paste(train_path, test_path):

    _, _, data_23 = load_values(train_path)

    _, _, data_24 = load_values(test_path)



    rows = min(data_23.shape[0], data_24.shape[0])

    cols = min(data_23.shape[1], data_24.shape[1])

    data_23 = data_23[:rows, :cols]

    data_24 = data_24[:rows, :cols]



    abs_diff = np.abs(data_23 - data_24)

    mae = float(abs_diff.mean())



    non_zero_mask = data_24 != 0

    if np.any(non_zero_mask):

        mape = float(np.abs((data_24[non_zero_mask] - data_23[non_zero_mask]) / data_24[non_zero_mask]).mean() * 100)

    else:

        mape = 0.0



    print(f"Aligned shape: {data_23.shape}")

    print(f"Mean Absolute Error (MAE): {mae:.4f} units")

    print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")



for train_set in train_sets:

    print(f"Evaluating copy-paste strategy for: {train_set}")

    calculate_copy_paste(train_set, test_set)

In [ ]:
def make_output_dir():

    candidates = [Path('outputs/kmedoid'), Path('notebooks/outputs/kmedoid')]

    for base in candidates:

        if base.parent.exists():

            base.mkdir(parents=True, exist_ok=True)

            return base

    fallback = Path('outputs/kmedoid')

    fallback.mkdir(parents=True, exist_ok=True)

    return fallback

In [ ]:
def dtw_distance(ts_a, ts_b):

    n, m = len(ts_a), len(ts_b)

    dp = np.full((n + 1, m + 1), np.inf)

    dp[0, 0] = 0.0

    for i in range(1, n + 1):

        for j in range(1, m + 1):

            cost = (ts_a[i - 1] - ts_b[j - 1]) ** 2

            dp[i, j] = cost + min(dp[i - 1, j], dp[i, j - 1], dp[i - 1, j - 1])

    return float(np.sqrt(dp[n, m]))



def build_distance_matrix(series_matrix):

    n = len(series_matrix)

    dist = np.zeros((n, n), dtype=float)

    for i in range(n):

        for j in range(i + 1, n):

            d = dtw_distance(series_matrix[i], series_matrix[j])

            dist[i, j] = dist[j, i] = d

    return dist



def k_medoids_from_dist(dist_matrix, n_clusters, random_state, max_iter=KMEDOID_MAX_ITER):

    rng_local = np.random.default_rng(random_state)

    n = dist_matrix.shape[0]

    medoids = rng_local.choice(n, size=n_clusters, replace=False)

    for _ in range(max_iter):

        labels = dist_matrix[:, medoids].argmin(axis=1)

        new_medoids = medoids.copy()

        for cluster in range(n_clusters):

            members = np.where(labels == cluster)[0]

            if len(members) == 0:

                candidates = np.setdiff1d(np.arange(n), new_medoids)

                if len(candidates) > 0:

                    new_medoids[cluster] = rng_local.choice(candidates)

                continue

            cluster_dist = dist_matrix[np.ix_(members, members)]

            new_medoids[cluster] = members[np.argmin(cluster_dist.sum(axis=1))]

        if np.array_equal(new_medoids, medoids):

            break

        medoids = new_medoids

    labels = dist_matrix[:, medoids].argmin(axis=1)

    return medoids, labels



def build_consensus(dist_matrix, n_clusters, n_runs=CONSENSUS_RUNS, sample_ratio=CONSENSUS_SAMPLE_RATIO, random_state=RANDOM_STATE):

    rng_local = np.random.default_rng(random_state)

    n = dist_matrix.shape[0]

    agreement = np.zeros((n, n), dtype=float)

    cooccurrence = np.zeros((n, n), dtype=float)

    for _ in range(n_runs):

        subset_size = max(n_clusters + 1, int(sample_ratio * n))

        subset_idx = np.sort(rng_local.choice(n, size=subset_size, replace=False))

        subset_dist = dist_matrix[np.ix_(subset_idx, subset_idx)]

        _, labels = k_medoids_from_dist(subset_dist, n_clusters, rng_local.integers(1 << 32))

        for i_pos, i in enumerate(subset_idx):

            for j_pos, j in enumerate(subset_idx):

                cooccurrence[i, j] += 1

                if labels[i_pos] == labels[j_pos]:

                    agreement[i, j] += 1

    similarity = np.divide(agreement, cooccurrence, out=np.zeros_like(agreement), where=cooccurrence > 0)

    np.fill_diagonal(similarity, 1.0)

    return similarity



def medoid_indices_from_labels(dist_matrix, labels):

    medoid_positions = []

    for cluster_id in sorted(np.unique(labels)):

        members = np.where(labels == cluster_id)[0]

        cluster_dist = dist_matrix[np.ix_(members, members)]

        medoid_positions.append(members[np.argmin(cluster_dist.sum(axis=1))])

    return medoid_positions

In [ ]:
def generate_clustering(ids, train_values):

    if SAMPLE_SIZE is not None:

        train_values = train_values[:SAMPLE_SIZE]

        ids = ids[:SAMPLE_SIZE]



    # Z-normalize

    mean = train_values.mean(axis=1, keepdims=True)

    std = train_values.std(axis=1, keepdims=True) + 1e-6

    norm_values = (train_values - mean) / std



    sample_size = min(600, len(norm_values))

    rng = np.random.default_rng(RANDOM_STATE)

    sample_pos = np.sort(rng.choice(len(norm_values), size=sample_size, replace=False))

    sample_ids = ids[sample_pos]

    sample_matrix = norm_values[sample_pos]



    print(f"Building DTW distance matrix for {sample_size} series...")

    dist_matrix = build_distance_matrix(sample_matrix)



    print("Running consensus clustering...")

    similarity = build_consensus(dist_matrix, n_clusters=CLUSTERS_COUNT, random_state=RANDOM_STATE)



    consensus_distance = 1.0 - similarity

    np.fill_diagonal(consensus_distance, 0.0)



    agg = AgglomerativeClustering(n_clusters=CLUSTERS_COUNT, metric='precomputed', linkage='average')

    sample_labels = agg.fit_predict(consensus_distance)



    final_medoid_positions = medoid_indices_from_labels(dist_matrix, sample_labels)

    final_medoid_ids = sample_ids[final_medoid_positions]



    id_to_pos = {id_val: idx for idx, id_val in enumerate(ids)}

    medoid_positions_full = [id_to_pos[mid] for mid in final_medoid_ids]

    medoid_vectors = norm_values[medoid_positions_full]



    diff = norm_values[:, None, :] - medoid_vectors[None, :, :]

    all_distances = np.sqrt((diff ** 2).sum(axis=2))

    full_labels = all_distances.argmin(axis=1)



    print(f"cluster_centers shape: {medoid_vectors.shape}")

    print(f"samples clustered: {len(full_labels)}")

    print(f"cluster sizes: {np.bincount(full_labels).tolist()}")



    return full_labels, medoid_vectors, norm_values

In [ ]:
output_dir = make_output_dir()

clusterings = []



for train_set in train_sets:

    print(f"Generating clustering for: {train_set}")

    ids, _, train_values = load_values(train_set)

    labels, medoid_vectors, norm_values = generate_clustering(ids, train_values)



    ids_for_export = ids[:SAMPLE_SIZE] if SAMPLE_SIZE is not None else ids



    export_df = pd.DataFrame({

        'ID': ids_for_export,

        'cluster': labels

    })

    export_name = f"kmedoid_cluster_labels_{Path(train_set).stem}.csv"

    export_path = output_dir / export_name

    export_df.to_csv(export_path, index=False)

    print(f"Saved: {export_path}")



    clusterings.append((train_set, labels, medoid_vectors, norm_values))

In [ ]:
import matplotlib.pyplot as plt



def print_clusters(labels, medoid_vectors, norm_values, plot_individual=PLOT_INDIVIDUAL_SERIES, max_series_per_cluster=MAX_SERIES_PER_CLUSTER_PLOT):

    optimal_k = len(medoid_vectors)



    fig, axes = plt.subplots(optimal_k, 1, figsize=(10, 3 * optimal_k), sharex=True)

    if optimal_k == 1:

        axes = [axes]



    for i in range(optimal_k):

        cluster_data = norm_values[labels == i]



        if plot_individual and len(cluster_data) > 0:

            sample_n = min(max_series_per_cluster, len(cluster_data))

            idx = np.random.choice(len(cluster_data), size=sample_n, replace=False)

            for series in cluster_data[idx]:

                axes[i].plot(series.ravel(), color='gray', alpha=0.08)



        axes[i].plot(medoid_vectors[i].ravel(), color='red', linewidth=3, label='Cluster Medoid')

        axes[i].set_title(f"Cluster {i+1} (Contains {len(cluster_data)} consumers)")

        axes[i].set_ylabel("Power Consumption")

        axes[i].legend()



    plt.xlabel("Days of the Year")

    plt.tight_layout()

    plt.show()



for train_set, labels, medoid_vectors, norm_values in clusterings:

    print(f"Plotting cluster panels for: {train_set}")

    print_clusters(labels, medoid_vectors, norm_values)

In [ ]:
import matplotlib.pyplot as plt



def print_all_clusters(labels, medoid_vectors, norm_values, plot_individual=PLOT_INDIVIDUAL_SERIES, max_series_per_cluster=MAX_SERIES_PER_CLUSTER_PLOT):

    optimal_k = len(medoid_vectors)

    colors = ['red', 'blue', 'green', 'orange', 'purple', 'cyan', 'brown', 'olive']



    plt.figure(figsize=(12, 6))



    for i in range(optimal_k):

        cluster_data = norm_values[labels == i]

        cluster_color = colors[i % len(colors)]



        if plot_individual and len(cluster_data) > 0:

            sample_n = min(max_series_per_cluster, len(cluster_data))

            idx = np.random.choice(len(cluster_data), size=sample_n, replace=False)

            for series in cluster_data[idx]:

                plt.plot(series.ravel(), color=cluster_color, alpha=0.04)



        plt.plot(

            medoid_vectors[i].ravel(),

            color=cluster_color,

            linewidth=3,

            label=f'Cluster {i+1} Medoid (n={len(cluster_data)})'

        )



    plt.title('k-Medoid Power Consumption Clusters')

    plt.xlabel('Days of the Year')

    plt.ylabel('Power Consumption')

    plt.legend(loc='upper right')

    plt.tight_layout()

    plt.show()



for train_set, labels, medoid_vectors, norm_values in clusterings:

    print(f"Plotting overlay clusters for: {train_set}")

    print_all_clusters(labels, medoid_vectors, norm_values)